In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()
DATA = Path("data")

## One position of a filter = one node: dot product, plus bias, then ReLU

In [ ]:
rng = np.random.default_rng(0)
x = rng.random((1, 6, 6, 1)).astype("float32")
conv = keras.layers.Conv2D(1, 3, activation="relu", bias_initializer=keras.initializers.Constant(0.1))
out = conv(x).numpy()[0, :, :, 0]
w, b = conv.get_weights()
window = x[0, 0:3, 0:3, 0].ravel()                 # the 9 pixels under the filter at the top-left
node = max(0.0, window @ w.ravel() + b[0])         # what an ANN node computes with these 9 inputs
print("Conv2D top-left value:", out[0, 0].round(5), "  node with 9 inputs:", round(float(node), 5))

## Parameters of a convolution layer do not depend on the image size

In [ ]:
rows = []
for side in (224, 1080):
    c = keras.layers.Conv2D(50, 3)
    c.build((None, side, side, 3))
    d = keras.layers.Dense(100)
    d.build((None, side * side * 3))
    rows.append(dict(image=f"{side} x {side} x 3", conv_50_filters=c.count_params(), dense_100_nodes=d.count_params()))
print(pd.DataFrame(rows).to_string(index=False))

## The comparison on real data
ANN: Flatten, Dense(128, relu), Dense(10, softmax), as in the MNIST ANN Note.
CNN: two Conv2D + MaxPooling blocks, Flatten, Dense(10, softmax).
Same optimizer, 15 epochs, batch 128; 3 seeds each; mean test accuracy.

In [ ]:
def ann():
    return keras.Sequential([keras.Input(shape=(28, 28, 1)), keras.layers.Flatten(),
                             keras.layers.Dense(128, activation="relu"), keras.layers.Dense(10, activation="softmax")])

def cnn():
    return keras.Sequential([keras.Input(shape=(28, 28, 1)),
                             keras.layers.Conv2D(16, 3, activation="relu"), keras.layers.MaxPooling2D(2),
                             keras.layers.Conv2D(32, 3, activation="relu"), keras.layers.MaxPooling2D(2),
                             keras.layers.Flatten(), keras.layers.Dense(10, activation="softmax")])

print("ANN parameters:", ann().count_params(), "  CNN parameters:", cnn().count_params())
for name, layers in (("ANN", [784 * 128, 128 * 10]), ("CNN", [26 * 26 * 16 * 9, 11 * 11 * 32 * 3 * 3 * 16, 800 * 10])):
    print(f"{name} multiplications per image: {sum(layers):,}")

In [ ]:
sets = {"MNIST": keras.datasets.mnist.load_data(), "Fashion-MNIST": keras.datasets.fashion_mnist.load_data()}
rows = []
for dname, ((xtr, ytr), (xte, yte)) in sets.items():
    xtr, xte = xtr[..., None] / 255.0, xte[..., None] / 255.0
    for mname, make in (("ANN", ann), ("CNN", cnn)):
        for seed in (1, 2, 3):
            keras.utils.set_random_seed(seed)
            m = make()
            m.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
            h = m.fit(xtr, ytr, epochs=15, batch_size=128, verbose=0)
            rows.append(dict(data=dname, model=mname, seed=seed, params=m.count_params(),
                             train_acc=h.history["accuracy"][-1], test_acc=m.evaluate(xte, yte, verbose=0)[1]))
res = pd.DataFrame(rows)
res.to_csv(DATA / "cnn_vs_ann.csv", index=False)
print(res.groupby(["data", "model"])[["params", "train_acc", "test_acc"]].mean().round(4))